In [ ]:
# @title 1 · Configure the single full-width run
# Run Cell 2 again before resuming Cell 3 after a runtime restart or finished evaluation.
ORIGINAL_CHECKPOINT = ""  # Leave blank to download the exact original FP16 model
WORK_ROOT = "/content/sd15_macaq_fullwidth_from_fp16"  # Set to a persistent path for cross-session resume
BACKEND = "auto"  # auto | emulated | native
ROWS_PER_CALL = 128
CACHE_LAYER_ON_GPU = True  # Upload the current layer’s samples once if VRAM permits
WIDTH = HEIGHT = 512
STEPS, CFG = 4, 1.2
NEGATIVE_PROMPT = ""
RELATIVE_FLIP_TOL = 1e-7  # Ignore numerical noise relative to the starting row error
GRAM_ROWS = 512
OUTPUT_ROW_CHUNK = 256
CONV_TILE_ROWS = 256
DOWNLOAD_REPORT = True  # Only the report ZIP downloads

import gc
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime.")
if BACKEND not in {"auto", "emulated", "native"}:
    raise ValueError("Invalid BACKEND")
cc = torch.cuda.get_device_capability()
backend = ("native" if cc[0] >= 10 else "emulated") if BACKEND == "auto" else BACKEND
if backend == "native" and cc[0] < 10:
    raise RuntimeError("Native FP4 needs a Blackwell GPU. Set BACKEND='emulated' on A100 or L4.")
if min(ROWS_PER_CALL, GRAM_ROWS, OUTPUT_ROW_CHUNK, CONV_TILE_ROWS) < 1 or not 0 < RELATIVE_FLIP_TOL < 1:
    raise ValueError("Use positive counts.")
packages = ["diffusers==0.39.0", "transformers==4.57.6", "accelerate>=1.0",
            "safetensors>=0.8", "huggingface_hub>=0.36,<1", "lpips==0.1.4", "ftfy", "requests"]
if importlib.util.find_spec("triton") is None:
    packages.append("triton>=3.6")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
import base64
import io
import zipfile

RUNTIME_BUNDLE = (
    '...'
    'AAAACQAJAH0CAADSewAAAAA='
)

RUNTIME_DIR = Path(WORK_ROOT).expanduser() / "runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(RUNTIME_BUNDLE))) as archive:
    archive.extractall(RUNTIME_DIR)
sys.path.insert(0, str(RUNTIME_DIR))
print(f"{torch.cuda.get_device_name()} · {backend} FP4 evaluation")


In [ ]:
# @title 2 · Download FP16 and cache inputs to all 282 layers
from collections import Counter, OrderedDict
import diffusers
import numpy as np
import requests
from torch import nn
from diffusers import LCMScheduler, StableDiffusionPipeline
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from macaq_unet_conv import patch_view, patch_rows, load_macaq_unet
from macaq_mxfp4 import balance_pre_scale_from_diag, get_bh, _SG
from macaq_mxfp4 import screen_block_exponents, pack_mxfp4_from_signed_index, rotate_blocks

FILENAME = "realismByStableYogi_sd15V9LCM.safetensors"
EXPECTED_SHA256 = "1a16a9524427019dc513b398087035e7e36ad2f2aa9691bd71d28ad959e94180"
MIRRORS = [
    ("Chroma111/CivitAI-Archive", f"166609/1235013/{FILENAME}"),
]
SD15_CONFIG = "stable-diffusion-v1-5/stable-diffusion-v1-5"

CALIBRATION_PROMPTS = [
    "Photorealistic headshot of an elderly man, window light, gray knit sweater",
    "Full-body editorial photo of an adult woman in a red coat on a city sidewalk",
    "Three friends talking over coffee in a busy cafe, candid documentary photo",
    "Golden retriever running through tall grass, low camera angle",
    "Dark forest with a small cabin, mist between pine trees, dawn",
    "Glass perfume bottle on wet black slate, studio product photography",
    "Busy produce market at midday, natural colors, wide angle street photography",
    "Portrait of a middle-aged woman with curly dark hair, warm side lighting",
    "Snowy mountain range reflected in a calm lake, landscape photography",
    "Neon-lit alley and parked motorcycle at night, rain, realistic photograph",
    "Close-up of a sliced orange and a kitchen knife on a wooden board",
    "Cozy apartment interior with a green velvet sofa, indirect afternoon light",
    "Cyclist crossing a concrete bridge, overcast weather, sports photography",
    "Adult fashion model in a tailored suit, full-length studio photograph",
    "Modern architecture with white facades and hard afternoon shadows",
    "A fishing boat on a gray sea, pastel clouds along the horizon",
    'Full-length street photograph of an adult woman in a beige trench coat, wind lifting the hem, cloudy daylight',
    'A studio headshot of an adult man looking left, black turtleneck, soft rim light and a neutral background',
    'Two ceramic mugs, one blue and one green, beside a small silver spoon on a white table',
    'A glass vase with three orange tulips beside a folded newspaper, morning light from the left',
    'An adult couple carrying shopping bags across a rainy pedestrian crossing, eye-level candid photo',
    'A black cat sitting on a windowsill beside a yellow book, light falling across its face',
    'An adult violinist on a theater stage, warm overhead spotlight, medium shot',
    'A wide photograph of an empty beach with a single red lifeguard chair at sunrise',
    'An adult baker arranging four rolls on a metal tray in a softly lit kitchen',
    'Side profile of a silver sports car under a concrete overpass, late afternoon photography',
    'An adult woman with short black hair wearing a green jacket in front of a pale brick wall',
    'A white bicycle leaning against a wooden fence, green hills in the background',
    'A glass of water to the left of a red apple and a blue cloth on a gray counter',
    "A close-up photograph of an adult man's hands repairing a mechanical wristwatch",
    'A portrait of an adult athlete wearing a navy hoodie outdoors in falling snow',
    'An interior photograph of a dim library with two leather chairs and a brass reading lamp',
]
EVALUATION = [
    ("Candid photo of an adult man holding a newspaper at a train station", 7001),
    ("A full-body photograph of an adult woman in a long blue coat at dusk", 7002),
    ("Close-up of a ceramic teapot, two cups, and a linen napkin on a table", 7003),
    ("A red vintage car parked on a curve in a desert highway, sunny afternoon", 7004),
    ("Portrait of an adult architect wearing black rectangular glasses in a sunlit studio", 7005),
    ("Exactly three green apples beside one yellow pear on a wooden board", 7006),
]

def file_sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def obtain_checkpoint():
    if ORIGINAL_CHECKPOINT:
        result = Path(ORIGINAL_CHECKPOINT).expanduser()
        if result.suffix.lower() != ".safetensors" or not result.is_file():
            raise ValueError("ORIGINAL_CHECKPOINT must point to the original .safetensors file.")
        if file_sha256(result) != EXPECTED_SHA256:
            raise ValueError("ORIGINAL_CHECKPOINT SHA-256 does not match Civitai version 1235013.")
        return result
    result = Path(WORK_ROOT).expanduser() / FILENAME
    result.parent.mkdir(parents=True, exist_ok=True)
    if result.is_file() and file_sha256(result) == EXPECTED_SHA256:
        return result
    previous = globals().get("CHECKPOINT_PATH")
    if previous:
        previous = Path(previous)
        if previous.is_file() and file_sha256(previous) == EXPECTED_SHA256:
            return previous
    for repo, filename in MIRRORS:
        try:
            candidate = Path(hf_hub_download(repo_id=repo, filename=filename))
        except Exception as exc:
            print(f"Could not download from {repo} ({type(exc).__name__}).")
            continue
        if file_sha256(candidate) == EXPECTED_SHA256:
            return candidate
        print(f"Skipping {repo} because its SHA-256 does not match version 1235013.")
    # Civitai may require an account token for the download.
    token = os.environ.get("CIVITAI_TOKEN")
    if not token:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                token = userdata.get("CIVITAI_TOKEN")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    temp = result.with_suffix(".partial")
    try:
        headers = {"Authorization": f"Bearer {token}"} if token else {}
        with requests.get("https://civitai.com/api/download/models/1235013",
                          headers=headers, stream=True, timeout=(20, 90)) as response:
            response.raise_for_status()
            with temp.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                    handle.write(chunk)
        if file_sha256(temp) != EXPECTED_SHA256:
            raise ValueError("Downloaded file SHA-256 did not match")
        temp.replace(result)
    except (requests.RequestException, ValueError) as exc:
        raise RuntimeError(
            "Automatic download failed. Download Civitai version 1235013 manually and set "
            "ORIGINAL_CHECKPOINT in Cell 1, or set the CIVITAI_TOKEN Colab secret."
        ) from exc
    finally:
        temp.unlink(missing_ok=True)
    return result

def make_pipeline(path):
    pipe = StableDiffusionPipeline.from_single_file(
        str(path), config=SD15_CONFIG, torch_dtype=torch.float16,
    ).to("cuda")
    if pipe.unet.config.in_channels != 4 or pipe.unet.config.cross_attention_dim != 768:
        raise RuntimeError("Expected a four-channel SD 1.5 U-Net with CLIP width 768.")
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config, clip_sample=False)
    pipe.set_progress_bar_config(disable=True)
    pipe.unet.eval()
    return pipe

@torch.inference_mode()
def generate(pipe, prompt, seed, output_type="pil", callback=None):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    result = pipe(
        prompt=prompt, negative_prompt=NEGATIVE_PROMPT or None,
        height=HEIGHT, width=WIDTH, guidance_scale=CFG, num_inference_steps=STEPS,
        generator=generator, output_type=output_type, callback_on_step_end=callback,
    )
    return result.images[0] if output_type == "pil" else result.images

CHECKPOINT_PATH = obtain_checkpoint()
if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
pipe = make_pipeline(CHECKPOINT_PATH)
device = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

specs = {}
for name, layer in pipe.unet.named_modules():
    if isinstance(layer, nn.Conv2d):
        if layer.groups != 1 or layer.padding_mode != "zeros" or isinstance(layer.padding, str):
            raise ValueError(f"Unsupported convolution geometry: {name}")
        spec = dict(kind="conv2d", in_features=math.prod(layer.weight.shape[1:]), out_features=layer.out_channels,
                    kernel=list(layer.kernel_size), stride=list(layer.stride), padding=list(layer.padding),
                    dilation=list(layer.dilation))
    elif isinstance(layer, nn.Linear):
        spec = dict(kind="linear", in_features=layer.in_features, out_features=layer.out_features)
    else:
        continue
    specs[name] = dict(**spec, weight_shape=list(layer.weight.shape), bias=layer.bias is not None)
original_unet_bytes = sum(t.numel() * t.element_size() for t in pipe.unet.state_dict().values())
expected_payload = sum(t.numel() * t.element_size() for key, t in pipe.unet.state_dict().items()
                       if not key.startswith(tuple(name + "." for name in specs)))
for spec in specs.values():
    padded = ((spec["in_features"] + 255) // 256) * 256
    output_rows = spec["out_features"]
    expected_payload += output_rows * (padded // 2 + padded // 32) + 2 * padded + 2 * 32 * 32
    if spec["bias"]:
        expected_payload += 2 * output_rows

counts_by_kind = Counter(spec["kind"] for spec in specs.values())
if counts_by_kind != {"conv2d": 98, "linear": 184}:
    raise ValueError(f"Unexpected SD 1.5 U-Net layer inventory: {dict(counts_by_kind)}")

calibration_seeds = list(range(4500, 4500 + len(CALIBRATION_PROMPTS)))
scheduler_config = dict(pipe.scheduler.config)
key = dict(source=EXPECTED_SHA256, layers=specs, prompts=CALIBRATION_PROMPTS, seeds=calibration_seeds,
           width=WIDTH, height=HEIGHT, steps=STEPS, cfg=CFG, negative=NEGATIVE_PROMPT,
           scheduler=scheduler_config, rows=ROWS_PER_CALL, torch=str(torch.__version__),
           gpu=torch.cuda.get_device_name(),
           runtime=hashlib.sha256((RUNTIME_DIR / "macaq_unet_conv.py").read_bytes()).hexdigest(),
           diffusers=diffusers.__version__, sampling_version=1)
work_id = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12]
work = Path(WORK_ROOT).expanduser() / work_id
cache_dir = work / "samples"
cache_dir.mkdir(parents=True, exist_ok=True)
complete = cache_dir / "complete.json"
sample_files = {name: cache_dir / f"{i:03d}.fp16" for i, name in enumerate(specs)}
max_calls = len(CALIBRATION_PROMPTS) * STEPS
sample_counts = json.loads(complete.read_text()) if complete.is_file() else {}
cache_ok = sample_counts.keys() == specs.keys() and all(
    sample_files[name].is_file() and sample_files[name].stat().st_size == n * specs[name]["in_features"] * 2
    for name, n in sample_counts.items()
)
if cache_ok:
    print("Reusing completed calibration inputs.")
else:
    complete.unlink(missing_ok=True)
    for path in sample_files.values():
        path.unlink(missing_ok=True)
    estimated = sum(s["in_features"] for s in specs.values()) * max_calls * ROWS_PER_CALL * 2
    free = shutil.disk_usage(cache_dir).free
    if free < estimated + 3 * 2**30:
        raise RuntimeError(f"Need up to {estimated / 2**30:.1f} GiB for samples plus 3 GiB for outputs. "
                           f"Free space is {free / 2**30:.1f} GiB under {cache_dir}.")
    print(f"Calibration cache upper bound: {estimated / 2**30:.2f} GiB.")
    counts, hits = Counter(), Counter()
    opened = OrderedDict()

    def capture(name):
        spec = specs[name]
        def hook(_, args):
            x = args[0]
            if spec["kind"] == "conv2d":
                view = patch_view(x, spec["kernel"], spec["stride"], spec["padding"], spec["dilation"])
                n = math.prod(view.shape[:3])
                idx = torch.linspace(0, n - 1, min(n, ROWS_PER_CALL), device=x.device).long()
                rows = patch_rows(view, idx)
            else:
                flat = x.reshape(-1, spec["in_features"])
                n = len(flat)
                idx = torch.linspace(0, n - 1, min(n, ROWS_PER_CALL), device=x.device).long()
                rows = flat.index_select(0, idx)
            rows = rows.to(device="cpu", dtype=torch.float16).contiguous()
            if name not in opened:
                if len(opened) == 32:
                    opened.popitem(last=False)[1].close()
                opened[name] = sample_files[name].open("ab")
            opened.move_to_end(name)
            opened[name].write(rows.numpy().tobytes())
            counts[name] += len(rows)
            hits[name] += 1
        return hook

    hooks = [pipe.unet.get_submodule(name).register_forward_pre_hook(capture(name)) for name in specs]
    try:
        for prompt, seed in tqdm(list(zip(CALIBRATION_PROMPTS, calibration_seeds, strict=True)),
                                 desc="Caching FP16 inputs"):
            generate(pipe, prompt, seed, output_type="latent")
    finally:
        for hook in hooks:
            hook.remove()
        for handle in opened.values():
            handle.close()
    missing = [name for name in specs if hits[name] != max_calls or counts[name] == 0]
    if missing:
        raise RuntimeError(f"Unexpected calibration coverage at {missing[0]}.")
    sample_counts = dict(counts)
    pending = complete.with_suffix(".tmp")
    pending.write_text(json.dumps(sample_counts, indent=2))
    pending.replace(complete)

print(f"Cached inputs for {len(specs)} layers. Largest width: {max(s['in_features'] for s in specs.values()):,}.")
print(f"Work directory: {work}")


In [ ]:
# @title 3 · Fit every layer to convergence, export, and compare to FP16
from PIL import Image, ImageDraw
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from IPython.display import display
from torch.nn import functional as F

fit_settings = dict(tolerance=RELATIVE_FLIP_TOL, gram_rows=GRAM_ROWS, output_chunk=OUTPUT_ROW_CHUNK,
                    version=1)
fit_tag = hashlib.sha256(json.dumps(fit_settings, sort_keys=True).encode()).hexdigest()[:12]
out = work / fit_tag
fitted = out / "fitted"
model_dir = out / "unet"
report = out / "report"
for directory in (fitted, model_dir, report):
    directory.mkdir(parents=True, exist_ok=True)


def rows_on_device(data, start, stop, device):
    if isinstance(data, torch.Tensor):
        return data[start:stop].float()
    return torch.from_numpy(np.array(data[start:stop], copy=True)).to(device, torch.float32)


@torch.inference_mode()
def gram_from_samples(data, n, k, pre, bh):
    h = torch.zeros((pre.numel(), pre.numel()), device=pre.device, dtype=torch.float32)
    for start in range(0, n, GRAM_ROWS):
        x = rows_on_device(data, start, start + GRAM_ROWS, pre.device)
        z = rotate_blocks(F.pad(x, (0, pre.numel() - k)) * pre, bh)
        h.addmm_(z.T, z)
    h.diagonal().add_(h.diagonal().mean())
    return h


@torch.inference_mode()
def refine_until_stable(codes, scales, target, h, progress=None):
    """Fit adjacent FP4 codes against the full Hessian at fixed block exponents."""
    ids = torch.arange(len(target), device=target.device)
    counts = torch.zeros_like(ids)
    starting_codes = codes.clone()
    levels = _SG.to(device=target.device, dtype=torch.float32)
    levels64 = levels.double()
    diagonal = h.diagonal()
    diagonal64 = diagonal.double()
    sweeps = 0

    def recompute(row_ids, precision):
        if precision == torch.float32:
            residual = target[row_ids] - levels[codes[row_ids]] * scales[row_ids]
            return residual, residual @ h
        residual = target[row_ids].double() - levels64[codes[row_ids]] * scales[row_ids].double()
        gradient = torch.empty_like(residual)
        # Accumulate over every input feature. Tile the output columns so
        # the full 23,040-wide Hessian never needs a second FP64 copy.
        for start in range(0, h.shape[1], 512):
            stop = min(start + 512, h.shape[1])
            gradient[:, start:stop] = residual @ h[:, start:stop].double().contiguous()
        return residual, gradient

    def moves(row_ids, gradient):
        high_precision = gradient.dtype == torch.float64
        grid, diag = (levels64, diagonal64) if high_precision else (levels, diagonal)
        current = codes[row_ids]
        step_scale = scales[row_ids].to(gradient.dtype)
        values = grid[current]
        up = (grid[(current + 1).clamp_max(len(grid) - 1)] - values) * step_scale
        down = (grid[(current - 1).clamp_min(0)] - values) * step_scale
        du = up.square() * diag - 2 * up * gradient
        dd = down.square() * diag - 2 * down * gradient
        du.masked_fill_(current == len(grid) - 1, math.inf)
        dd.masked_fill_(current == 0, math.inf)
        loss, col = torch.minimum(du, dd).min(dim=1)
        up_loss = du.gather(1, col[:, None])
        down_loss = dd.gather(1, col[:, None])
        upward = up_loss <= down_loss
        delta = torch.where(upward, up.gather(1, col[:, None]), down.gather(1, col[:, None]))
        return loss, col, upward.squeeze(1), delta.squeeze(1)

    residual, gradient = recompute(ids, torch.float32)
    initial_loss = (gradient * residual).sum(1).clamp_min(0)
    if not torch.isfinite(initial_loss).all():
        raise RuntimeError("Nonfinite initial objective. Check the calibration inputs.")
    tolerance = (initial_loss * RELATIVE_FLIP_TOL).clamp_min(1e-12)

    def walk(row_ids, gradient):
        nonlocal sweeps
        precision = gradient.dtype
        since_refresh = 0
        while row_ids.numel():
            loss, col, upward, delta = moves(row_ids, gradient)
            keep = loss < -tolerance[row_ids]
            row_ids, col, upward, delta = row_ids[keep], col[keep], upward[keep], delta[keep]
            if not row_ids.numel():
                return
            gradient = gradient[keep]
            codes[row_ids, col] += upward.long() * 2 - 1
            counts[row_ids] += 1
            gradient -= h.index_select(0, col).to(precision) * delta[:, None]
            sweeps += 1
            since_refresh += 1
            if progress is not None and sweeps % 1024 == 0:
                progress(sweeps, row_ids.numel())
            if since_refresh == 128:
                _, gradient = recompute(row_ids, precision)
                since_refresh = 0

    walk(ids, gradient)
    residual, gradient = recompute(ids, torch.float32)
    final_loss = (residual * gradient).sum(1)
    best_remaining, _, _, _ = moves(ids, gradient)
    if not (torch.isfinite(final_loss).all() and torch.isfinite(best_remaining).all()):
        raise RuntimeError("Nonfinite convergence objective. Check the calibration inputs.")

    increased = final_loss > initial_loss + tolerance * 8
    pending = ids[(best_remaining < -tolerance) | increased]
    for batch in pending.split(16):
        if not batch.numel():
            continue
        # Restart any rows whose recomputed objective increased. Continue the
        # others from their current codes, with the same convergence tolerance.
        restart = batch[increased[batch]]
        codes[restart] = starting_codes[restart]
        counts[restart] = 0

        # Compute the comparison loss from the original codes in FP64 as well.
        current = codes[batch]
        codes[batch] = starting_codes[batch]
        orig_residual, orig_gradient = recompute(batch, torch.float64)
        original_exact = (orig_residual * orig_gradient).sum(1)
        codes[batch] = current

        while True:
            residual, gradient = recompute(batch, torch.float64)
            remaining, _, _, _ = moves(batch, gradient)
            if not torch.isfinite(remaining).all():
                raise RuntimeError("Nonfinite FP64 convergence objective.")
            improve = remaining < -tolerance[batch]
            if not improve.any():
                break
            walk(batch[improve], gradient[improve])

        corrected = (residual * gradient).sum(1)
        if not torch.isfinite(corrected).all() or (corrected > original_exact + tolerance[batch] * 8).any():
            raise RuntimeError("FP64 refinement increased a row's objective. Check the calibration inputs.")
        final_loss[batch] = corrected.float()
    return counts.cpu(), float(initial_loss.sum()), float(final_loss.sum())


@torch.inference_mode()
def pack_full_width(weight, bias, path, n, layer_name):
    weight = weight.flatten(1).float()
    k = weight.shape[1]
    padded = ((k + 31) // 32) * 32
    weight = F.pad(weight, (0, padded - k))
    data = np.memmap(path, dtype=np.float16, mode="r", shape=(n, k))
    cached = False
    if weight.is_cuda and CACHE_LAYER_ON_GPU:
        free, _ = torch.cuda.mem_get_info()
        estimate = n * k * 2 + padded**2 * 4 + 4 * 2**30
        if free > estimate:
            try:
                # Copy-on-write map stays on disk and is only read during CUDA upload.
                mapped = np.memmap(path, dtype=np.float16, mode="c", shape=(n, k))
                data = torch.from_numpy(mapped).to(weight.device)
                del mapped
                cached = True
            except torch.OutOfMemoryError:
                torch.cuda.empty_cache()
    diagonal = torch.zeros(k, device=weight.device)
    for start in range(0, n, GRAM_ROWS):
        x = rows_on_device(data, start, start + GRAM_ROWS, weight.device)
        diagonal.add_(x.square().sum(0))
    del x
    pre = balance_pre_scale_from_diag(weight, F.pad(diagonal, (0, padded - k)), n, alpha=.75)
    if not torch.isfinite(pre).all():
        raise RuntimeError(f"Nonfinite calibration scale at {layer_name}.")
    bh = get_bh(32, weight.device)
    h = gram_from_samples(data, n, k, pre, bh.float())
    del data
    packed_rows, scale_rows, flip_counts = [], [], []
    starting_error = final_error = 0.0
    for i, chunk in enumerate(weight.split(OUTPUT_ROW_CHUNK, dim=0), 1):
        rotated = rotate_blocks(chunk / pre, bh.float())
        codes, exponents, scales = screen_block_exponents(rotated, h.diagonal(), 32)
        flips, before, after = refine_until_stable(
            codes, scales, rotated, h,
            progress=lambda sweeps, remaining: print(
                f"{layer_name}, chunk {i}: {sweeps:,} sweeps, {remaining} active rows", flush=True
            ),
        )
        packed, blocks = pack_mxfp4_from_signed_index(codes, exponents, 32)
        packed_rows.append(packed.cpu())
        scale_rows.append(blocks.view(torch.uint8)[:len(chunk), :padded // 32].cpu())
        flip_counts.append(flips)
        starting_error += before
        final_error += after
    pad = (-padded) % 256
    result = {
        "W_packed": F.pad(torch.cat(packed_rows), (0, pad // 2)),
        "W_scales": F.pad(torch.cat(scale_rows), (0, pad // 32), value=127),
        "pre_scale": F.pad(pre.bfloat16().cpu(), (0, pad), value=1),
        "B_inv_T": bh.cpu(),
    }
    if bias is not None:
        result["bias"] = bias.detach().bfloat16().cpu()
    flips = torch.cat(flip_counts)
    log = dict(input_width=k, hessian_width=padded, samples=n, empirical_rank_upper_bound=min(n, k),
               rows_converged=len(flips), mean_flips_per_row=flips.float().mean().item(),
               max_flips_per_row=flips.max().item(), screened_objective=starting_error,
               converged_objective=final_error, samples_cached_on_gpu=cached)
    return {key: value.contiguous() for key, value in result.items()}, log


if pipe.unet is None or not isinstance(pipe.unet.get_submodule(next(iter(specs))), (nn.Conv2d, nn.Linear)):
    raise RuntimeError("Rerun Cell 2 to reload FP16 before resuming this cell.")
torch.cuda.reset_peak_memory_stats()
layer_log = {}
print(f"Found {sum((fitted / (name + '.safetensors')).is_file() for name in specs)}/{len(specs)} saved layer fits.")
for name in tqdm(sorted(specs, key=lambda n: (specs[n]["in_features"], n)), desc="Fitting all layers"):
    path = fitted / f"{name}.safetensors"
    if path.is_file():
        with safe_open(str(path), framework="pt") as existing:
            log = json.loads(existing.metadata()["fit"])
        spec = specs[name]
        if (log["input_width"], log["samples"], log["rows_converged"]) != (
            spec["in_features"], sample_counts[name], spec["out_features"],
        ):
            raise ValueError(f"Fit checkpoint metadata mismatch at {name}. Remove {path} and rerun Cell 3.")
        layer_log[name] = log
        continue
    layer = pipe.unet.get_submodule(name)
    start = time.monotonic()
    print(f"{name} · {specs[name]['in_features']:,} inputs", flush=True)
    state, log = pack_full_width(layer.weight, layer.bias, sample_files[name], sample_counts[name], name)
    torch.cuda.synchronize()
    log["seconds"] = time.monotonic() - start
    log["kind"] = specs[name]["kind"]
    pending = path.with_suffix(".tmp")
    save_file(state, str(pending), metadata={"fit": json.dumps(log)})
    pending.replace(path)
    layer_log[name] = log
    del layer, state
    print(f"  converged · {log['max_flips_per_row']:,} max flips/row · {log['seconds']:.1f} s", flush=True)

peak_fit_gib = torch.cuda.max_memory_allocated() / 2**30
prefixes = tuple(name + "." for name in specs)
state = {name: value.detach().cpu().contiguous()
         for name, value in pipe.unet.state_dict().items() if not name.startswith(prefixes)}
for name in specs:
    state.update({f"{name}.{k}": v for k, v in load_file(str(fitted / f"{name}.safetensors")).items()})
payload = sum(value.numel() * value.element_size() for value in state.values())
if payload != expected_payload:
    raise RuntimeError(f"Packed payload is {payload:,} bytes, expected {expected_payload:,}.")
pending = model_dir / "model.tmp"
save_file(state, str(pending))
pending.replace(model_dir / "model.safetensors")
del state
pipe.unet.save_config(model_dir)
shutil.copytree(RUNTIME_DIR, model_dir / "runtime", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
manifest = dict(
    format="sd15-macaq-full-unet-conv-v1", source_file=FILENAME, source_sha256=EXPECTED_SHA256,
    conv_hessian_block="full_per_layer", linear_hessian_block="full_per_layer",
    layers=specs, unet_tensor_bytes=payload, scheduler_config=scheduler_config, steps=STEPS,
    cfg=CFG, width=WIDTH, height=HEIGHT, negative_prompt=NEGATIVE_PROMPT,
    calibration_prompts=CALIBRATION_PROMPTS, calibration_seeds=calibration_seeds,
    rows_per_step=ROWS_PER_CALL, fit_settings=fit_settings, original_unet_parameter_bytes=original_unet_bytes,
    conv_tile_rows=CONV_TILE_ROWS, layer_fits=layer_log,
)
(model_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n")
print(f"Saved {payload / 1e6:.2f} MB of packed U-Net tensors to {model_dir / 'model.safetensors'}")

fp16_images = [generate(pipe, p, s) for p, s in tqdm(EVALUATION, desc="FP16 references")]
pipe.unet = None
gc.collect()
torch.cuda.empty_cache()
pipe.unet = load_macaq_unet(model_dir, backend=backend, device=device, tile_rows=CONV_TILE_ROWS)
packed_images = [generate(pipe, p, s) for p, s in tqdm(EVALUATION, desc="Full-width W4A4")]
import lpips
metric = lpips.LPIPS(net="alex", verbose=False).to(device).eval()

def image_tensor(image):
    return torch.from_numpy(np.array(image.convert("RGB"), dtype=np.float32)).to(device).permute(2, 0, 1)[None] / 127.5 - 1

rows = []
with torch.inference_mode():
    for i, ((prompt, seed), ref, packed) in enumerate(zip(EVALUATION, fp16_images, packed_images, strict=True), 1):
        board = Image.new("RGB", (2 * WIDTH, HEIGHT + 24), "#151515")
        board.paste(ref, (0, 24))
        board.paste(packed, (WIDTH, 24))
        draw = ImageDraw.Draw(board)
        draw.text((8, 5), "FP16", fill="white")
        draw.text((WIDTH + 8, 5), "Full-width MACaQ W4A4", fill="white")
        filename = f"pair_{i:02d}.png"
        board.save(report / filename)
        rows.append(dict(image=filename, seed=seed, prompt=prompt,
                         lpips_to_fp16=metric(image_tensor(ref), image_tensor(packed)).item()))
summary = dict(
    unet_tensor_bytes=payload, original_unet_bytes=original_unet_bytes, packed_layers=len(specs),
    runtime=backend, gpu=torch.cuda.get_device_name(), width=WIDTH, height=HEIGHT, steps=STEPS, cfg=CFG,
    calibration_prompts=CALIBRATION_PROMPTS, fitting_peak_allocated_gib=peak_fit_gib,
    rows_per_step=ROWS_PER_CALL, fit_settings=fit_settings, layer_fits=layer_log,
    mean_lpips_to_fp16=sum(row["lpips_to_fp16"] for row in rows) / len(rows), images=rows,
)
(report / "fullwidth_from_fp16.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")
print(f"Mean LPIPS to FP16: {summary['mean_lpips_to_fp16']:.4f}")
display(Image.open(report / rows[0]["image"]))
archive = shutil.make_archive(str(out / "sd15_macaq_fullwidth_from_fp16_report"), "zip", report)
print(f"Report ZIP: {archive}")
if DOWNLOAD_REPORT:
    try:
        from google.colab import files
    except ImportError:
        pass
    else:
        files.download(archive)
